In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Extract)=
# Extract

*Extracting a subset of atoms and structures from a molecular system.*

The function {func}`molsysmt.basic.extract` allows you to extract a subset of a molecular system based on atom selections and structure indices. Its output is a new molecular system with the elements and structures selected by the user. A composite input, such as a topology file paired with a coordinate trajectory, is materialized as a native `molsysmt.MolSys` before extraction so both domains remain aligned. For an H5MSM file input, provide a different `output_filename` to write a subset, or request `to_form='molsysmt.MolSys'` to keep the subset in memory.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.basic.extract`.
:::

For an explicit hierarchy selection, pass `element="group"`, `"chain"`,
`"molecule"` or `"entity"`; atomistic systems retain the selected elements' atoms.
Native inventories without atoms instead retain their declared groups and
parents. Their default axis is groups, and explicit chain selection can retain
empty chains. Requested group order is preserved, memberships are remapped and
repeated extraction indices are rejected. Source-preserving string extraction
keeps its existing sequence-position contract. See
{ref}`Cookbook_Sequence_source_groups` for a small native example.


## Basic usage

Let's show how this function works using a PDB structure (`181L`):


In [2]:
import molsysmt as msm

In [3]:
molsys = msm.convert('181L')

In [4]:
msm.info(molsys, element='entity')

index,name,type,n atoms,n groups,n components,n chains,n molecules
0,T4 LYSOZYME,protein,1289,162,1,1,1
1,CHLORIDE ION,ion,2,2,2,2,2
2,2-HYDROXYETHYL DISULFIDE,small molecule,8,1,1,1,1
3,BENZENE,small molecule,6,1,1,1,1
4,water,water,136,136,136,1,136


We will extract only the protein and benzene entities from the system by using a selection based on `entity_name`.
For ViewerJSON, an explicit atom list preserves your order and remaps retained
bonds and their aligned order/type columns. Repeated atom indices are rejected.
Structure lists preserve order and repetitions. Selected graphics data are copied
after selection; discarded structures are not copied. See
{ref}`ViewerJSON <user-foundations-native-world-classes-molsysmt-viewerjson>`.


In [5]:
molsubsys = msm.extract(molsys, selection='entity_name in ["T4 LYSOZYME", "BENZENE"]')

:::{tip}
:class: dropdown

All methods defined in the {ref}`molsysmt.basic <API basic>` module can also be invoked from the main level of the library. Hence, {func}`molsysmt.extract` is the same function as {func}`molsysmt.basic.extract`.
:::

In [6]:
msm.info(molsubsys, element='entity')

index,name,type,n atoms,n groups,n components,n chains,n molecules
0,T4 LYSOZYME,protein,1289,162,1,1,1
1,BENZENE,small molecule,6,1,1,1,1


## Restricting chemical connectivity

Native extraction restricts each chemical state independently. Complete stored
connectivity gives components of the retained induced graph. Removing the middle
atom of a three-atom path leaves two components, while stable molecule ownership
can remain one. A changed component gets a local string ID and unknown name/type;
only unchanged full source atom sets retain their labels.

The example below explicitly declares complete connectivity in the public typed
chemical-state record. This is supplied evidence, not a completeness assessment
performed by extraction. With partial or unavailable connectivity, extraction
instead restricts declared membership and downgrades cut complete component
declarations to partial. It does not prepare a standalone fragment or add caps
and hydrogens. See {ref}`Cookbook_Group_Free_Hierarchy` for both cases.


In [7]:
from molsysmt.native import MolSys

molsys_A = MolSys(n_atoms=3, n_molecules=1)
molsys_A.topology.set_atom_molecules([0, 0, 0])
molsys_A.topology.add_bonds([[0, 1], [1, 2]])
state_records = msm.convert(molsys_A.chemical_states, to_form='molsysmt.ChemicalStatesDict')
state_records.data['states'][0]['connectivity_completeness'] = 'complete'
molsys_A.chemical_states = msm.convert(state_records, to_form='molsysmt.ChemicalStates')
molsys_B = msm.extract(molsys_A, selection=[0, 2])
print(msm.get(molsys_B, n_atoms=True, n_components=True, n_molecules=True))
print(msm.get(molsys_B, element='atom', molecule_index=True))
print(msm.get(molsys_A, n_bonds=True, n_components=True))

[2, 2, 1]
[0, 0]
[2, 1]


:::{seealso}
:class: dropdown

**Related Tools & References**

- {ref}`Tutorial_Convert`: Convert a molecular system into another form or forms with {func}`molsysmt.basic.convert`.
- {ref}`Tutorial_Info`: Print a summary of the contents, topology, and structural data of a molecular system with {func}`molsysmt.basic.info`.
:::